In [ ]:
import pandas as pd
import sqlite3


# Cargar las bases1
df_edu = pd.read_csv("C:/Users/erick/primer_repo_izainea/Datos/MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR__B_SICA_Y_MEDIA_POR_MUNICIPIO_20250623.csv", encoding="latin1")
df_muni = pd.read_csv("C:/Users/erick/primer_repo_izainea/Datos/DIVIPOLA-_C_digos_municipios_20250623.csv", encoding="latin1")

df_edu.columns = df_edu.columns.str.encode('latin1').str.decode('utf-8')
df_muni.columns = df_muni.columns.str.encode('latin1').str.decode('utf-8')
# Vista preliminar
df_edu.head(), df_muni.head()


(    AÑO  CÓDIGO_MUNICIPIO    MUNICIPIO  CÓDIGO_DEPARTAMENTO DEPARTAMENTO  \
 0  2023              5001    MedellÃ­n                    5    Antioquia   
 1  2023              5002    Abejorral                    5    Antioquia   
 2  2023              5004    AbriaquÃ­                    5    Antioquia   
 3  2023              5021  AlejandrÃ­a                    5    Antioquia   
 4  2023              5030       AmagÃ¡                    5    Antioquia   
 
    CÓDIGO_ETC              ETC POBLACIÓN_5_16  TASA_MATRICULACIÓN_5_16  \
 0      3759.0        MedellÃ­n         377562                    96.15   
 1      3758.0  Antioquia (ETC)           3634                    74.38   
 2      3758.0  Antioquia (ETC)            503                    62.62   
 3      3758.0  Antioquia (ETC)            864                    81.37   
 4      3758.0  Antioquia (ETC)           5060                    78.30   
 
    COBERTURA_NETA  ...  REPROBACIÓN  REPROBACIÓN_TRANSICIÓN  \
 0           95.94  

In [63]:
# Selección de columnas
df_edu_clean = df_edu[[
    'AÑO', 'CÓDIGO_MUNICIPIO', 'MUNICIPIO',
    'CÓDIGO_DEPARTAMENTO', 'DEPARTAMENTO',
    'POBLACIÓN_5_16', 'TASA_MATRICULACIÓN_5_16',
    'COBERTURA_NETA', 'APROBACIÓN', 'REPROBACIÓN', 'REPITENCIA'
]]

# Renombrar columnas
df_edu_clean = df_edu_clean.rename(columns={
    'AÑO': 'anio',
    'CÓDIGO_MUNICIPIO': 'cod_municipio',
    'MUNICIPIO': 'municipio',
    'CÓDIGO_DEPARTAMENTO': 'cod_departamento',
    'DEPARTAMENTO': 'departamento',
    'POBLACIÓN_5_16': 'poblacion_objetivo',
    'TASA_MATRICULACIÓN_5_16': 'tasa_matricula',
    'COBERTURA_NETA': 'cobertura_neta',
    'APROBACIÓN': 'aprobacion',
    'REPROBACIÓN': 'reprobacion',
    'REPITENCIA': 'repitencia'
})
df_edu.head(), df_muni.head()


(    AÑO  CÓDIGO_MUNICIPIO    MUNICIPIO  CÓDIGO_DEPARTAMENTO DEPARTAMENTO  \
 0  2023              5001    MedellÃ­n                    5    Antioquia   
 1  2023              5002    Abejorral                    5    Antioquia   
 2  2023              5004    AbriaquÃ­                    5    Antioquia   
 3  2023              5021  AlejandrÃ­a                    5    Antioquia   
 4  2023              5030       AmagÃ¡                    5    Antioquia   
 
    CÓDIGO_ETC              ETC POBLACIÓN_5_16  TASA_MATRICULACIÓN_5_16  \
 0      3759.0        MedellÃ­n         377562                    96.15   
 1      3758.0  Antioquia (ETC)           3634                    74.38   
 2      3758.0  Antioquia (ETC)            503                    62.62   
 3      3758.0  Antioquia (ETC)            864                    81.37   
 4      3758.0  Antioquia (ETC)           5060                    78.30   
 
    COBERTURA_NETA  ...  REPROBACIÓN  REPROBACIÓN_TRANSICIÓN  \
 0           95.94  

In [64]:
# Convertir columnas numéricas
cols_num = ['anio', 'cod_municipio', 'cod_departamento', 'poblacion_objetivo',
            'tasa_matricula', 'cobertura_neta', 'aprobacion', 'reprobacion', 'repitencia']

for col in cols_num:
    df_edu_clean[col] = pd.to_numeric(df_edu_clean[col], errors='coerce')

# Eliminar valores faltantes
df_edu_clean.dropna(inplace=True)

# Convertir nombres de municipios a mayúsculas para evitar conflictos
df_edu_clean['municipio'] = df_edu_clean['municipio'].str.upper()
df_edu_clean['departamento'] = df_edu_clean['departamento'].str.upper()


In [66]:
df_edu_clean['dep_mun'] = df_edu_clean['departamento'] + '-' + df_edu_clean['municipio']

# Crear DataFrame limpio de municipios
df_muni['dep_mun'] = df_muni['Nombre Departamento'].str.upper() + '-' + df_muni['Nombre Municipio'].str.upper()
df_muni_clean = df_muni[['Código Departamento', 'Código Municipio', 'dep_mun', 'Latitud', 'longitud']].copy()
df_muni_clean.columns = ['cod_departamento', 'cod_municipio', 'dep_mun', 'latitud', 'longitud']


In [67]:
# Comprobación de unión (verás si hay nulls que debes resolver)
df_merged = df_edu_clean.merge(df_muni_clean, on='dep_mun', how='left')
df_merged[['dep_mun', 'latitud', 'longitud']].isnull().sum()


dep_mun        0
latitud     7868
longitud    7868
dtype: int64

In [68]:
# Preparar llave para unión
df_muni['dep_mun'] = df_muni['Nombre Departamento'].str.upper() + '-' + df_muni['Nombre Municipio'].str.upper()

# Selección y renombrado
df_muni_clean = df_muni[['Código Departamento', 'Código Municipio', 'dep_mun', 'Latitud', 'longitud']].copy()
df_muni_clean.columns = ['cod_departamento', 'cod_municipio', 'dep_mun', 'latitud', 'longitud']


In [69]:
dim_tiempo = df_edu_clean[['anio']].drop_duplicates().sort_values(by='anio').reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

In [70]:
dim_geo = df_edu_clean[['dep_mun', 'cod_departamento', 'cod_municipio']].drop_duplicates().reset_index(drop=True)
dim_geo = dim_geo.merge(df_muni_clean, on='dep_mun', how='left')
dim_geo['id_geografia'] = dim_geo.index + 1


In [72]:
# Agrupar por rangos de desempeño como dimensiones analíticas
def clasificar_rendimiento(valor):
    if valor >= 95:
        return 'ALTO'
    elif valor >= 75:
        return 'MEDIO'
    else:
        return 'BAJO'

df_edu_clean['nivel_aprobacion'] = df_edu_clean['aprobacion'].apply(clasificar_rendimiento)
df_edu_clean['nivel_cobertura'] = df_edu_clean['cobertura_neta'].apply(clasificar_rendimiento)
df_edu_clean['nivel_matricula'] = df_edu_clean['tasa_matricula'].apply(clasificar_rendimiento)

dim_educativa = df_edu_clean[['nivel_aprobacion', 'nivel_cobertura', 'nivel_matricula']].drop_duplicates().reset_index(drop=True)
dim_educativa['id_nivel_educativo'] = dim_educativa.index + 1


In [73]:
# Merge con claves subrogadas
fact = df_edu_clean.merge(dim_tiempo, on='anio', how='left')
fact = fact.merge(dim_geo[['dep_mun', 'id_geografia']], on='dep_mun', how='left')
fact = fact.merge(dim_educativa, on=['nivel_aprobacion', 'nivel_cobertura', 'nivel_matricula'], how='left')

# Tabla de hechos final
fact_matriculas = fact[[
    'id_tiempo', 'id_geografia', 'id_nivel_educativo',
    'poblacion_objetivo', 'tasa_matricula', 'cobertura_neta',
    'aprobacion', 'reprobacion', 'repitencia'
]]


In [74]:
import sqlite3

# Crear base de datos SQLite
conn = sqlite3.connect("modelo_estrella_educacion.db")

# Exportar tablas
dim_tiempo.to_sql("Dim_Tiempo", conn, if_exists="replace", index=False)
dim_geo.to_sql("Dim_Geografia", conn, if_exists="replace", index=False)
dim_educativa.to_sql("Dim_Educativa", conn, if_exists="replace", index=False)
fact_matriculas.to_sql("Fact_Matriculas", conn, if_exists="replace", index=False)

conn.close()


In [75]:
import sqlite3
import pandas as pd

# Conectar base de datos creada en la fase anterior
conn = sqlite3.connect("modelo_estrella_educacion.db") 



Para este análisis se eligió un modelo de datos en estrella debido a su estructura sencilla y eficiente para consultas analíticas, lo que facilita la exploración de indicadores educativos en diferentes dimensiones. El modelo se compone de una tabla de hechos principal (`Fact_Matriculas`) que centraliza las métricas cuantitativas, y tres dimensiones: `Dim_Geografia`, `Dim_Tiempo` y `Dim_Educativa`, las cuales permiten analizar la cobertura educativa por ubicación, periodo temporal y tipo de institución, respectivamente. Esta configuración fue suficiente para responder las preguntas planteadas, sin necesidad de agregar una cuarta dimensión, ya que la información requerida se pudo descomponer y calcular adecuadamente desde estas tres perspectivas clave.


1. ¿Qué porcentaje de escolaridad hay respecto a la población?

In [76]:

# Conexión a la base de datos
conn = sqlite3.connect("modelo_estrella_educacion.db")

# Consulta al último año disponible
query = """
SELECT 
    G.dep_mun AS municipio,
    T.anio,
    F.poblacion_objetivo,
    F.cobertura_neta
FROM Fact_Matriculas F
JOIN Dim_Tiempo T ON F.id_tiempo = T.id_tiempo
JOIN Dim_Geografia G ON F.id_geografia = G.id_geografia
WHERE T.anio = (SELECT MAX(anio) FROM Dim_Tiempo)
  AND F.cobertura_neta IS NOT NULL
  AND F.poblacion_objetivo IS NOT NULL
"""

# Ejecutar consulta
df = pd.read_sql_query(query, conn)

# Calcular matriculados estimados
df['matriculados_estimados'] = (df['poblacion_objetivo'] * df['cobertura_neta']) / 100

# Calcular porcentaje de escolaridad real
df['porcentaje_escolaridad_valor'] = (df['matriculados_estimados'] / df['poblacion_objetivo']) * 100

# Aplicar tope lógico de 100%
df['porcentaje_escolaridad_valor'] = df['porcentaje_escolaridad_valor'].clip(upper=100)

# Formatear como porcentaje visible
df['porcentaje_escolaridad'] = df['porcentaje_escolaridad_valor'].map('{:.2f}%'.format)

# Ordenar de mayor a menor
df = df.sort_values(by='porcentaje_escolaridad_valor', ascending=False)

# Mostrar tabla final
display(df[['municipio', 'anio', 'poblacion_objetivo', 'matriculados_estimados', 'porcentaje_escolaridad']])




,municipio,anio,poblacion_objetivo,matriculados_estimados,porcentaje_escolaridad
1053,ARAUCA-TAME,2023,12148.0,13766.1136,100.00%
1047,ARAUCA-ARAUCA,2023,19694.0,19879.1236,100.00%
984,TOLIMA-MELGAR,2023,6512.0,7949.8496,100.00%
1045,VALLE DEL CAUCA-YUMBO,2023,20596.0,21539.2968,100.00%
998,TOLIMA-SAN LUIS,2023,2261.0,2310.0637,100.00%
...,...,...,...,...,...
1110,GUAVIARE-MIRAFLORES,2023,2014.0,654.9528,32.52%
751,NARIÃ±O-MAGÃ¼I,2023,7399.0,2259.6546,30.54%
1114,VAUPÃ©S-TARAIRA,2023,814.0,183.9640,22.60%
1092,AMAZONAS-LA VICTORIA,2023,169.0,23.9980,14.20%


Para responder esta pregunta se estimó el **porcentaje de escolaridad por municipio** calculando cuántos niños, niñas y adolescentes de la **población objetivo** (entre 5 y 16 años) efectivamente se encuentran matriculados en el sistema educativo.

Esta estimación se realizó utilizando la **cobertura neta** reportada por el Ministerio de Educación, que indica el porcentaje de la población en edad escolar que se encuentra matriculada en el grado correspondiente.

 Se construyó una métrica ajustada en la que:

 $$
 \text{Porcentaje de escolaridad} = \frac{\text{Matriculados estimados}}{\text{Población objetivo}} \times 100
 $$

Los resultados muestran una importante variación entre municipios. Algunos superan el **90% de escolaridad**, lo cual evidencia una muy buena cobertura, mientras que otros apenas alcanzan porcentajes por debajo del **50%**, indicando posibles barreras de acceso, deserción o limitaciones en la oferta educativa.

Para asegurar la validez de los resultados, se estableció un **tope máximo del 100%**, ya que no es lógico tener más estudiantes matriculados que población en edad escolar. Los valores superiores a este umbral fueron ajustados, lo cual permitió interpretar la escolaridad de manera más realista y coherente.

Este análisis permite a las autoridades identificar zonas con **alta efectividad educativa** y otras que requieren **acciones urgentes en materia de cobertura, acceso o permanencia escolar**.



2. ¿Cómo compararía el rendimiento educativo por municipios?

In [77]:
query2 = """
SELECT 
    G.dep_mun AS municipio,
    ROUND(AVG(F.aprobacion), 2) AS aprobacion_promedio,
    ROUND(AVG(F.reprobacion), 2) AS reprobacion_promedio,
    ROUND(AVG(F.repitencia), 2) AS repitencia_promedio
FROM Fact_Matriculas F
JOIN Dim_Geografia G ON F.id_geografia = G.id_geografia
GROUP BY G.dep_mun
ORDER BY aprobacion_promedio DESC
"""

df_rendimiento = pd.read_sql_query(query2, conn)
display(df_rendimiento)


,municipio,aprobacion_promedio,reprobacion_promedio,repitencia_promedio
0,AMAZONAS-LA VICTORIA,100.00,0.00,14.95
1,AMAZONAS-PUERTO ARICA,99.20,0.04,7.43
2,CUNDINAMARCA-TIBIRITA,98.42,0.43,0.83
3,CUNDINAMARCA-LA PALMA,98.28,0.20,1.58
4,CUNDINAMARCA-GUATAVITA,98.27,0.27,2.62
...,...,...,...,...
1121,GUAINÃ­A-PUERTO COLOMBIA,75.68,13.93,5.02
1122,GUAINÃ­A-CACAHUAL,75.59,14.58,5.86
1123,GUAINÃ­A-MAPIRIPANA,73.32,17.80,7.74
1124,NACIONAL-NACIONAL,0.90,0.07,0.04


El rendimiento educativo se evaluó mediante tres indicadores clave: aprobación, reprobación y repitencia.

Los municipios con altos niveles de aprobación (superiores al 95%) y bajos niveles de repitencia y reprobación se destacan como regiones con mejor desempeño en la gestión escolar y procesos de aprendizaje.

Municipios como Tibirita, La Palma y Guatavita (Cundinamarca) muestran una combinación sólida de alta aprobación y baja repitencia, lo que sugiere entornos de aprendizaje efectivos y posiblemente un buen acompañamiento pedagógico.

Por otro lado, niveles elevados de repitencia pueden estar asociados a dificultades de retención escolar, calidad docente desigual o factores socioeconómicos limitantes.

3. ¿Qué departamentos tienen mejor cobertura?

In [82]:
query3 = """
SELECT 
    SUBSTR(G.dep_mun, 1, INSTR(G.dep_mun, '-') - 1) AS departamento,
    ROUND(AVG(F.cobertura_neta), 2) AS cobertura_promedio
FROM Fact_Matriculas F
JOIN Dim_Geografia G ON F.id_geografia = G.id_geografia
GROUP BY departamento
ORDER BY cobertura_promedio DESC
"""

df_cobertura = pd.read_sql_query(query3, conn)
display(df_cobertura)



,departamento,cobertura_promedio
0,BOGOTÃ¡ D.C.,95.89
1,QUINDIO,94.58
2,CESAR,93.92
3,SUCRE,93.80
4,MAGDALENA,93.32
5,META,90.74
6,TOLIMA,89.24
7,CASANARE,88.64
8,CUNDINAMARCA,88.61
9,"BOGOTÃ¡, D.C.",88.55


Al analizar la cobertura neta promedio por departamento, se observa que Bogotá D.C., Quindío y Cesar presentan los niveles más altos de cobertura educativa en el país, superando el 93%.

Esto refleja una infraestructura educativa consolidada, acceso garantizado y políticas de cobertura efectivas. Estos departamentos pueden servir como modelo de referencia para mejorar las condiciones en otras regiones.

La alta cobertura también puede estar relacionada con una menor dispersión rural, mayor disponibilidad de instituciones, y programas de matrícula o permanencia escolar.

4. Análisis de desempeño educativo según clasificación (Dim_Educativa)

In [79]:
query4 = """
SELECT 
    E.nivel_aprobacion,
    E.nivel_cobertura,
    E.nivel_matricula,
    COUNT(*) AS total_ocurrencias
FROM Fact_Matriculas F
JOIN Dim_Educativa E ON F.id_nivel_educativo = E.id_nivel_educativo
GROUP BY E.nivel_aprobacion, E.nivel_cobertura, E.nivel_matricula
ORDER BY total_ocurrencias DESC
"""

df_clasificacion = pd.read_sql_query(query4, conn)
display(df_clasificacion)

,nivel_aprobacion,nivel_cobertura,nivel_matricula,total_ocurrencias
0,MEDIO,MEDIO,MEDIO,4912
1,MEDIO,ALTO,ALTO,2449
2,MEDIO,BAJO,BAJO,2006
3,ALTO,MEDIO,MEDIO,1891
4,ALTO,BAJO,BAJO,951
5,ALTO,ALTO,ALTO,766
6,ALTO,MEDIO,BAJO,272
7,MEDIO,MEDIO,BAJO,241
8,MEDIO,MEDIO,ALTO,237
9,ALTO,MEDIO,ALTO,196


Si se activa la dimensión Dim_Educativa, se puede observar cómo se distribuyen los municipios entre tres niveles de clasificación: alto, medio y bajo, en función de la aprobación, matrícula y cobertura.

La mayoría de los municipios se concentran en el rango medio, lo cual indica que existe una oportunidad de mejora, especialmente en las regiones clasificadas como de nivel bajo.

Esta clasificación permite identificar zonas prioritarias para intervenciones pedagógicas o políticas educativas diferenciadas

5. Evolución temporal por año y departamento

In [80]:
query5 = """
SELECT 
    T.anio,
    SUBSTR(G.dep_mun, 1, INSTR(G.dep_mun, '-') - 1) AS departamento,
    ROUND(AVG(F.tasa_matricula), 2) AS tasa_matricula_promedio,
    ROUND(AVG(F.aprobacion), 2) AS aprobacion_promedio
FROM Fact_Matriculas F
JOIN Dim_Tiempo T ON F.id_tiempo = T.id_tiempo
JOIN Dim_Geografia G ON F.id_geografia = G.id_geografia
GROUP BY T.anio, departamento
ORDER BY T.anio, aprobacion_promedio DESC
"""

df_evolucion = pd.read_sql_query(query5, conn)
display(df_evolucion)

,anio,departamento,tasa_matricula_promedio,aprobacion_promedio
0,2011,ARAUCA,73.45,97.61
1,2011,"ARCHIPIÃ©LAGO DE SAN ANDRÃ©S, PROVIDENCIA Y SA...",75.27,97.10
2,2011,CHOCÃ³,80.52,96.57
3,2011,CUNDINAMARCA,88.44,96.47
4,2011,BOLÃ­VAR,85.74,96.29
...,...,...,...,...
424,2023,QUINDIO,95.13,86.63
425,2023,GUAVIARE,65.17,85.02
426,2023,VICHADA,93.55,81.39
427,2023,GUAINÃ­A,62.71,79.70


El análisis temporal muestra tendencias estables en la mayoría de los departamentos. Se destacan años en los que ciertos departamentos, como Arauca y Cundinamarca, presentaron picos de aprobación superiores al 96%.

A nivel nacional, la tasa de matrícula ha mostrado fluctuaciones moderadas, posiblemente asociadas a factores externos como políticas de gratuidad, condiciones económicas, o incluso eventos como la pandemia.

Este análisis permite realizar seguimiento a políticas educativas a lo largo del tiempo y evaluar si los esfuerzos institucionales se reflejan en mejoras sostenidas.

In [32]:
conn.close()